# **Applied Statistics for Data Science**

Este notebook acompanha o Capítulo 7 do livro e apresenta, na prática,
como aplicar estatística em projetos reais de Data Science.

Você aprenderá a:
- Formular problemas estatísticos
- Executar EDA avançado
- Aplicar testes estatísticos
- Rodar regressões aplicadas
- Integrar estatística com Machine Learning
- Interpretar resultados para tomada de decisão

Tempo estimado: **60–90 minutos**

## **Objectives**

Após completar este lab, você será capaz de:

* Realizar EDA avançado  
* Aplicar testes estatísticos em problemas reais  
* Usar regressão para inferência e previsão  
* Integrar estatística com modelos de Machine Learning  
* Interpretar resultados estatísticos no contexto de negócio  
* Executar estudos de caso completos  

---
## **Import Libraries**

Todas as Libraries necessárias estão listadas abaixo.
Os termos técnicos permanecem em inglês.

In [ ]:
!pip install pandas
!pip install numpy
!pip install scipy
!pip install seaborn
!pip install matplotlib
!pip install statsmodels

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

## **Load Dataset**

Usaremos o mesmo dataset utilizado nos capítulos anteriores para manter consistência pedagógica.

In [ ]:
ratings_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/teachingratings.csv'
df = pd.read_csv(ratings_url)

# **1. Applied EDA (Exploratory Data Analysis)**

Nesta seção, realizamos uma exploração avançada dos dados.

## **1.1 Overview do Dataset**

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

## **1.2 Detectando Missing Values**

In [ ]:
df.isna().sum()

## **1.3 Distribuições das variáveis numéricas**

In [ ]:
df.hist(figsize=(12, 8), bins=20)
plt.tight_layout()
plt.show()

## **1.4 Correlação entre variáveis contínuas**

In [ ]:
sns.heatmap(df.corr(), annot=True, cmap="Blues")
plt.show()

---
# **2. Applied Hypothesis Testing**

Aqui aplicamos testes estatísticos em problemas reais.

## **2.1 T-test: Gender afeta teaching evaluation?**

*H0:* não há diferença  
*H1:* existe diferença  

In [ ]:
scipy.stats.ttest_ind(df[df.gender=="female"]["eval"],
                      df[df.gender=="male"]["eval"],
                      equal_var=True)

## **Conclusão:**  
Se p-value < 0.05 → rejeitamos H0.

## **2.2 ANOVA: Beauty score difere por age group?**

In [ ]:
df.loc[df.age <= 40, "age_group"] = "≤40"
df.loc[(df.age > 40) & (df.age < 57), "age_group"] = "40–57"
df.loc[df.age >= 57, "age_group"] = "≥57"

In [ ]:
scipy.stats.f_oneway(df[df.age_group=="≤40"]["beauty"],
                     df[df.age_group=="40–57"]["beauty"],
                     df[df.age_group=="≥57"]["beauty"])

## **Conclusão:**  
Se p-value < 0.05 → pelo menos um grupo difere.

## **2.3 Chi-square: Existe associação entre tenure e gender?**

In [ ]:
cont = pd.crosstab(df.tenure, df.gender)
scipy.stats.chi2_contingency(cont)

---
# **3. Applied Regression Analysis**

Agora aplicamos regressão para inferência e previsão.

## **3.1 Simple Regression: Beauty → Eval**

In [ ]:
X = sm.add_constant(df["beauty"])
y = df["eval"]

model = sm.OLS(y, X).fit()
model.summary()

## **Conclusão:**  
β₁ indica o impacto marginal de beauty no evaluation score.

## **3.2 Multiple Regression**

Vamos incluir variáveis explicativas adicionais.

In [ ]:
X = df[["beauty", "age", "students"]]
X = sm.add_constant(X)
y = df["eval"]

model2 = sm.OLS(y, X).fit()
model2.summary()

## **Conclusão:**  
Avalie:
- significância dos coeficientes  
- direção do efeito  
- magnitude  
- R²  

---
# **4. Applied Feature Engineering**

Criando variáveis úteis para modelagem.

## **4.1 Dummy Variables**

In [ ]:
df_dummies = pd.get_dummies(df, columns=["gender", "tenure", "division"], drop_first=True)
df_dummies.head()

## **4.2 Interaction Terms**

Exemplo: interação entre beauty e gender.

In [ ]:
df["beauty_gender_interaction"] = df["beauty"] * df["female"]

In [ ]:
sm.OLS(df["eval"], sm.add_constant(df[["beauty", "female", "beauty_gender_interaction"]])).fit().summary()

---
# **5. Applied Machine Learning Integration**

Aqui mostramos como estatística fundamenta ML.

## **5.1 Train/Test Split**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

X = df[["beauty", "age", "students"]]
y = df["eval"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## **5.2 Train Model**

In [ ]:
ml_model = LinearRegression()
ml_model.fit(X_train, y_train)

In [ ]:
preds = ml_model.predict(X_test)
mean_squared_error(y_test, preds)

## **Conclusão:**  
O MSE indica o erro médio de previsão.

---
# **6. Case Study: Churn-style Problem Adaptado**

Vamos simular um problema aplicado usando variáveis do dataset.

## **Pergunta:**  
*Quais variáveis explicam melhor teaching evaluation score?*

In [ ]:
X = df_dummies.drop(columns=["eval"])
y = df["eval"]

X = sm.add_constant(X)
case_model = sm.OLS(y, X).fit()
case_model.summary()

## **Conclusões do Estudo de Caso**

- Identifique variáveis significativas (p < 0.05)  
- Avalie direção e magnitude dos coeficientes  
- Avalie R² e R² ajustado  
- Interprete no contexto educacional  

---
# **7. Practice Questions**

Exercícios para fixação.

### **Q1.** Beauty score difere entre male e female?  
*Use regression analysis.*

In [ ]:
## insert code here

### **Q2.** Age prediz evaluation score?  
*Use simple regression.*

In [ ]:
## insert code here

### **Q3.** Existe correlação entre students e beauty?  
*Use Pearson correlation.*

In [ ]:
## insert code here

### **Q4.** Crie um modelo preditivo completo para evaluation score.  
*Use regressão múltipla com dummies.*

In [ ]:
## insert code here

---
# **End of Applied Statistics for Data Science Lab**

Excelente trabalho!  
Você concluiu o laboratório prático mais avançado do livro.